In [5]:
import pandas as pd

# Lead Table - DIM
df_leads = pd.read_csv(
    "DTC Leads Data Set - dim lead(Sheet1).csv",
    parse_dates=[
        'Lead Created (ET) Time',
        'Last Contact Attempt (ET) Time'
    ],
)

# Call Activity Table - FACT

df_calls = pd.read_csv(
    "Call Activity Data Set - Dials(Call Activity Data Set 2026-02-).csv",
    parse_dates=[
        'Lead Created (ET) Time',
        'Activity (ET) Time',
        'First Contact (ET) Time'
    ]
)

# Shape
print("Leads shape:".ljust(18), df_leads.shape)
print("Calls shape:".ljust(18), df_calls.shape)

print("\nLEADS COLS")
print(df_leads.dtypes)

print("\nCALL ACTIVITY COLS")
print(df_calls.dtypes)

Leads shape:       (35170, 19)
Calls shape:       (382454, 23)

LEADS COLS
Lead ID                                          str
Last Contact Attempt (ET) Time        datetime64[us]
Utm Source                                       str
Utm Medium                                       str
Lead Source                                      str
Lead Channel Segment                             str
Borrower Requested Loan Type                     str
Current Lead Status                              str
Lead Created (ET) Date                           str
Lead Created (ET) Time                datetime64[us]
Gross Leads                                    int64
Number of Called Leads                         int64
Number of Contacted Leads                      int64
Number of Opportunities                        int64
Number of File Started Leads                   int64
Number of Entered Processing Leads             int64
Number of Locked Loan Leads                    int64
Number of Pre-Approval L

In [6]:
# Join check

print("Lead table join key sample:")
print(df_leads['Lead ID'].head(3))
print(df_leads['Lead ID'].dtype)
print()

print("Call activity table join key sample:")
print(df_calls['Lead ID'].head(3))
print(df_calls['Lead ID'].dtype)
print()

print("Any missing values in join key?")
print("leads  :", df_leads['Lead ID'].isna().sum())
print("calls  :", df_calls['Lead ID'].isna().sum())

Lead table join key sample:
0    00QPh00000iEOItMAO
1    00QPh00000evzVxMAI
2    00QPh00000cZP7KMAW
Name: Lead ID, dtype: str
str

Call activity table join key sample:
0    00QPh00000kH72cMAC
1    00QPh00000kH72cMAC
2    00QPh00000kH72cMAC
Name: Lead ID, dtype: str
str

Any missing values in join key?
leads  : 1
calls  : 0


In [7]:
# PRE-FILTER: Leads that haven't been called due to business requirement conditions

# 1. How many leads have NULL last contact attempt?
null_last_attempt = df_leads['Last Contact Attempt (ET) Time'].isna().sum()
total_leads = len(df_leads)

print(f"Number of leads with NO contact attempt: {null_last_attempt:,}")
print(f" == {null_last_attempt / total_leads:.1%} of all Leads\n")

# 2. Quick distribution of called vs not called
print("Breakdown:")
print(df_leads['Last Contact Attempt (ET) Time'].notna().value_counts(normalize=True).mul(100).round(1).astype(str) + " %")
print()

# 3. Just to be extra sure...are there any empty strings instead of real NaT?
print("# of empty strings that look like missing?")
print((df_leads['Last Contact Attempt (ET) Time'].astype(str).str.strip() == '').sum())

Number of leads with NO contact attempt: 4,154
 == 11.8% of all Leads

Breakdown:
Last Contact Attempt (ET) Time
True     88.2 %
False    11.8 %
Name: proportion, dtype: str

# of empty strings that look like missing?
0


In [8]:
# Original size
total_leads = len(df_leads)
print(f"Original leads: {total_leads:,}")

# PRE-FILTER - keeping only leads that were actually called
df_leads_called = df_leads[
    df_leads['Last Contact Attempt (ET) Time'].notna()
].copy()

called_leads = len(df_leads_called)
print(f"Leads with at least one contact attempt: {called_leads:,}")
print(f" == {called_leads / total_leads:.1%} of total Leads\n")

# SANITY CHECK - last attempt time should now have 0 nulls
print("# of NULSS in Last Contact Attempt (ET) Time after filter:")
print(df_leads_called['Last Contact Attempt (ET) Time'].isna().sum())

Original leads: 35,170
Leads with at least one contact attempt: 31,016
 == 88.2% of total Leads

# of NULSS in Last Contact Attempt (ET) Time after filter:
0


In [ ]:
# Do an INNER JOIN - clean standardized dataframe to work with

# Drop the redundant columns confirmed as identical
redundant_cols = ['Lead Created (ET) Time', 'Lead Created (ET) Date']

df_calls_clean = df_calls.drop(
    columns=[c for c in redundant_cols if c in df_calls.columns],
    errors='ignore'
)

print(f"Dropped {len(redundant_cols)} redundant columns from calls table")
print("Calls columns now:", df_calls_clean.columns.tolist()[:10], "...") 

# Standardize join key (just in case)
df_leads_called['Lead ID'] = df_leads_called['Lead ID'].astype(str).str.strip()
df_calls_clean['Lead ID']  = df_calls_clean['Lead ID'].astype(str).str.strip()

# INNER JOIN - only leads + calls that match on Lead ID

df_merged = pd.merge(
    df_leads_called,
    df_calls_clean,
    on='Lead ID',
    how='inner'
)

# Basic post-merge sanity checks
print("\nMerged shape:".ljust(25), df_merged.shape)
print("Unique leads in merged:".ljust(25), df_merged['Lead ID'].nunique())
print("Total call activity rows:".ljust(25), len(df_merged))
print("Avg calls per lead:".ljust(25), round(len(df_merged) / df_merged['Lead ID'].nunique(), 2))

; # Check if any suffixes were actually needed
; suffix_cols = [c for c in df_merged.columns if c.endswith('_lead') or c.endswith('_call')]
; print("\nColumns that received suffixes:", suffix_cols if suffix_cols else "None — very clean!")

; # Quick peek
; print("\nFirst 3 rows of merged data (selected interesting columns):")
; interesting_cols = [
;     'Lead ID',
;     'Lead Created (ET) Time',
;     'Last Contact Attempt (ET) Time',
;     'Activity (ET) Time',
;     'Outcome',
;     'Contacted?',
;     'Call Attempt Number',
;     'Is Latest Dial Attempt?',
;     'Current Lead Status',
;     'Lead Channel Segment'
; ]
; print(df_merged[interesting_cols].head(3).to_string(index=False))

Merged shape:        (382387, 41)
Unique leads in merged: 30169
Unique call activities: 382387

Columns from leads side (with _lead suffix where needed):
['Lead ID', 'Last Contact Attempt (ET) Time', 'Utm Source', 'Utm Medium', 'Lead Source', 'Lead Channel Segment', 'Borrower Requested Loan Type', 'Current Lead Status', 'Lead Created (ET) Date_lead', 'Lead Created (ET) Time_lead', 'Gross Leads', 'Number of Called Leads', 'Number of Contacted Leads', 'Number of Opportunities', 'Number of File Started Leads', 'Number of Entered Processing Leads', 'Number of Locked Loan Leads', 'Number of Pre-Approval Leads', 'Number of Funded Loan Leads']

Columns from calls side (with _call suffix where needed):
['Lead Created (ET) Date_call', 'Lead Created (ET) Time_call', 'Activity (ET) Time', 'Outcome', 'Call Attempt Number']

First 2 rows of merged data:
              Lead ID Last Contact Attempt (ET) Time Utm Source Utm Medium                  Lead Source Lead Channel Segment Borrower Requested Loa

In [26]:
# 1. Find columns that exist in both (case-sensitive, exact name match)
common_cols = set(df_leads_called.columns) & set(df_calls.columns)
common_cols = common_cols - {'Lead ID'}   # remove the join key

print("Columns that exist in BOTH tables (excluding Lead ID):")
print(sorted(common_cols))
print()

# 2. If there are any common columns, do a quick check on a sample
if common_cols:
    print("Quick check — are they identical? (sample of 5000 rows)")
    
    # temporary small inner join just for checking
    temp_merged = pd.merge(
        df_leads_called[['Lead ID'] + list(common_cols)],
        df_calls[['Lead ID'] + list(common_cols)],
        on='Lead ID',
        how='inner',
        suffixes=('_lead', '_call')
    )
    
    for col in common_cols:
        lead_col = f"{col}_lead"
        call_col = f"{col}_call"
        
        # are they equal where both exist?
        both_notna = temp_merged[[lead_col, call_col]].notna().all(axis=1)
        match_rate = (temp_merged.loc[both_notna, lead_col] == temp_merged.loc[both_notna, call_col]).mean()
        
        print(f"  {col:.<35} match rate: {match_rate:.4%}  "
              f"(based on {both_notna.sum():,} matching non-null pairs)")
else:
    print("→ No overlapping columns besides Lead ID → no need to worry about duplicates")

Columns that exist in BOTH tables (excluding Lead ID):
['Lead Created (ET) Date', 'Lead Created (ET) Time']

Quick check — are they identical? (sample of 5000 rows)
  Lead Created (ET) Time............. match rate: 100.0000%  (based on 382,387 matching non-null pairs)
  Lead Created (ET) Date............. match rate: 100.0000%  (based on 382,387 matching non-null pairs)
